In [ ]:
import pandas as pd

# Lectura de los archivos crudos
print("--- Iniciando Motor de Conciliación ETL ---")
df_subdiario = pd.read_csv('Subdiario_Ventas.csv')
df_contabilidad = pd.read_csv('Mayor_Contable.csv')

print(f"Total registros Subdiario: {len(df_subdiario)}")
print(f"Total registros Contabilidad: {len(df_contabilidad)}") 

--- Iniciando Motor de Conciliación ETL ---
Total registros Subdiario: 5
Total registros Contabilidad: 5


In [ ]:
# 1. Limpieza de CUITs (Eliminacion de guiones para que coincidan)
df_subdiario['CUIT'] = df_subdiario['CUIT'].str.replace('-', '')
df_contabilidad['CUIT_Cliente'] = df_contabilidad['CUIT_Cliente'].astype(str).str.replace('-', '')

# 2. Homologación de columnas (Renombrarlas para que el sistema las pueda cruzar)
df_contabilidad = df_contabilidad.rename(columns={
    'Comprobante': 'Factura', 
    'CUIT_Cliente': 'CUIT'
})

print("Datos limpios y homologados. Listos para cruzar.")

Datos limpios y homologados. Listos para cruzar.


In [ ]:
# Outer Join para ver qué hay en ambas bases y qué quedó solo
df_cruce = pd.merge(df_subdiario, df_contabilidad, on=['Factura', 'CUIT'], how='outer', indicator=True)

# Filtro 1: Facturas contabilizadas que NO existen en el sistema operativo (Riesgo Alto)
faltante_operativo = df_cruce[df_cruce['_merge'] == 'right_only'].copy()

# Filtro 2: Facturas emitidas NO contabilizadas (Evasión/Olvido)
faltante_contable = df_cruce[df_cruce['_merge'] == 'left_only'].copy()

# Filtro 3: Diferencias de montos en facturas que sí cruzaron
df_coincidentes = df_cruce[df_cruce['_merge'] == 'both'].copy()
df_coincidentes['Diferencia_Cruce'] = df_coincidentes['Monto_Contable'] - df_coincidentes['Monto_Operativo']
errores_monto = df_coincidentes[df_coincidentes['Diferencia_Cruce'] != 0].copy()

# En pantalla el error de monto 
print("\n--- ¡ALERTA: Diferencias de Monto Detectadas! ---")
display(errores_monto[['Factura', 'CUIT', 'Monto_Operativo', 'Monto_Contable', 'Diferencia_Cruce']])


--- ¡ALERTA: Diferencias de Monto Detectadas! ---


,Factura,CUIT,Monto_Operativo,Monto_Contable,Diferencia_Cruce
1,A-0002,30222222222,200500.0,200000.0,-500.0


In [ ]:
# Generar archivo Excel con múltiples pestañas ordenadas
with pd.ExcelWriter('Reporte_Excepciones_Auditoria.xlsx') as writer:
    faltante_operativo.to_excel(writer, sheet_name='Contabilizado_Sin_Soporte', index=False)
    faltante_contable.to_excel(writer, sheet_name='Emitido_No_Contabilizado', index=False)
    errores_monto.to_excel(writer, sheet_name='Diferencias_de_Monto', index=False)

print("\nReporte de excepciones exportado con éxito a Excel.")


Reporte de excepciones exportado con éxito a Excel.


In [7]:
# 5. Resumen Ejecutivo para la consola
print("\n" + "="*40)
print("   RESUMEN EJECUTIVO DE AUDITORÍA")
print("="*40)
print(f"Total de registros cruzados: {len(df_cruce)}")
print(f" Coincidencias exactas: {len(df_coincidentes) - len(errores_monto)}")
print(f" Facturas con diferencias de monto: {len(errores_monto)}")
print(f" Riesgo Alto - Sin soporte operativo: {len(faltante_operativo)}")
print(f" Riesgo Fiscal - No contabilizadas: {len(faltante_contable)}")
print("="*40)


   RESUMEN EJECUTIVO DE AUDITORÍA
Total de registros cruzados: 6
 Coincidencias exactas: 3
 Facturas con diferencias de monto: 1
 Riesgo Alto - Sin soporte operativo: 1
 Riesgo Fiscal - No contabilizadas: 1
